# **Lab 2 – Tidying, Cleaning, Imputation, and Outlier Detection**

## Machine Learning Programming

This assignment continues the work started in **`week5Lab.ipynb`**.  
Open this notebook and complete the activities **in this notebook**, preserving the section order and the overall structure.

The purpose of this lab is to practice the data-preprocessing work that normally takes place before machine learning: reshaping data, cleaning inconsistent values, handling missing data, detecting outliers, visualizing results, and explaining the decisions you make.

## **Assignment Overview**

### Purpose
By the end of this lab, you should be able to take several imperfect datasets and make them more suitable for analysis and later machine-learning work.

### Learning Outcomes
You will practice how to:

- inspect the shape, columns, data types, and quality of a dataset;
- distinguish **wide** and **long/tidy** data;
- reshape data with `pd.melt()` or `DataFrame.melt()`;
- clean strings using Pandas string methods;
- identify, quantify, drop, and impute missing values;
- compare mean, median, mode, and `SimpleImputer` approaches;
- identify outliers visually with box plots and scatter plots;
- identify outliers statistically using **Z-Scores** and **IQR**;
- justify preprocessing decisions using evidence from the data;
- document a reproducible Python data-processing workflow.

### Estimated Completion Time
Approximately **2.5–3.5 hours**, depending on your Python experience.

## **Submission and Project Structure**

Keep the project organized so another person can reproduce your work.

Recommended structure:

```text
project/
├── week5Lab.ipynb
├── README.md
├── requirements.txt
├── .gitignore
└── data/
    ├── pew-raw.csv
    ├── billboard.csv
    ├── cars.csv
    └── diabetes.csv
```

A clearly named equivalent data folder such as `CSVs/` or `datasets/` is also acceptable if all notebook paths are updated consistently.

### Required submission artifacts

- `week5Lab.ipynb` containing **all four completed sections**
- `README.md` with at least 1–2 meaningful sentences describing the lab and how to run it
- `requirements.txt` containing the packages required to reproduce the environment
- `.gitignore`
- all required data files inside a clearly named data folder

**Important:** Use relative paths such as `./data/cars.csv`. Do not use absolute paths from your own computer.

**Environment practice:** Use `requirements.txt` to define the environment. Do not add `!pip install ...` cells to the notebook when a requirements file is provided.

## **Before You Begin**

Use the following libraries where appropriate:

- `pandas`
- `numpy`
- `matplotlib`
- `seaborn`
- `scikit-learn`

Run the import cell below before beginning the exercises.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer

# Optional display settings
pd.set_option("display.max_columns", None)

## **General Expectations**

For every section:

1. Write the required Python code.
2. Display enough output to demonstrate that the code works.
3. Add short Markdown explanations in your own words.
4. Label plots clearly with a title and axis labels.
5. Explain what the output means instead of only producing it.
6. Preserve the required section headings so the notebook is easy to follow.
7. Before submitting, restart the kernel and use **Run All** to confirm the notebook executes from beginning to end.

# **Section 1 – PEW Research Dataset**

## **Learning Topic: Data Tidying and Reshaping**

A dataset is commonly described as **tidy** when:

1. each variable forms a column;
2. each observation forms a row; and
3. each type of observational unit forms a table.

The PEW dataset represents counts for religions across income ranges. In its original form, income values appear as column headings. You will reshape the dataset from **wide** format to **long/tidy** format using `melt()`.

### Required concepts
- dataset inspection
- wide versus long data
- identifier variables
- measured variables
- `pd.melt()` or `DataFrame.melt()`
- visualization after reshaping

## Getting the PEW Research Dataset

For this lab, the original workshop dataset has been replaced with the Pew Research survey dataset.

The dataset contains 36,908 rows and 654 columns. Each row represents a survey respondent. The columns contain respondent identifiers, survey responses, demographic and religious information, social-attitude variables, and survey-weighting information.

### Dataset Location

The CSV files is stored in the project's `data/` folder.

The expected project structure is:

```text
Lab2 - Tidying, Cleaning, Imputation, and Outlier Detection/
├── week5Lab.ipynb
├── requirements.txt
├── README.md
├── .gitignore

└── data/
    └── README.md
    └── billboard.csv
    └── Pew_Religious_dataset .csv
```

### **Task 1.1 – Load and Inspect the Dataset**

1. Load the PEW dataset from your data folder.
2. Display its shape.
3. Display its column names.
4. Display several rows using `head()`, `loc`, `iloc`, or `tail()`.
5. Identify the column that should remain the identifier variable.

In [ ]:
# Task 1.1: Load and inspect the Pew Research dataset

from pathlib import Path

# Locate the CSV using a relative path
file_path = Path("data/Pew_Religious_dataset .csv")

# Load the dataset
df_pew = pd.read_csv(file_path, low_memory=False)

# Display the dataset dimensions
print("Dataset dimensions:", df_pew.shape)

# Display the column names
print("\nColumn names:")
print(df_pew.columns.tolist())

# Display the first five rows
print("\nFirst five rows:")
display(df_pew.head())

# Check the respondent identifier
identifier_column = "P_SUID"

print("\nIdentifier column:", identifier_column)
print("Unique identifiers:", df_pew[identifier_column].nunique())
print("Missing identifiers:", df_pew[identifier_column].isna().sum())
print("Duplicate identifiers:", df_pew[identifier_column].duplicated().sum())

### **Task 1.2 – Explain the Problem**

In a Markdown cell below, answer:

- What is not tidy about the original PEW dataset?
- Which values are stored in column names?
- Which column should be treated as the identifier (`id_vars`)?

# Task 1.2 – Understanding the Dataset Structure

### 1. What is not tidy about the dataset?

The Pew Research dataset contains 36,908 observations and 654 columns. Each row represents a survey respondent, and the columns contain respondent identifiers, survey responses, demographic information, and other recorded variables.

The dataset uses a wide format, with different survey variables stored in separate columns. However, a wide format is not automatically untidy. To determine whether reshaping is appropriate, we must examine how the variables relate to one another and what analysis we want to perform.

For this assignment, we will select appropriate survey variables and reshape them into a long format to make comparisons between the selected variables easier.

### 2. Which values are stored in column names?

The column names identify the variables recorded in the survey. Examples include `HAPPY`, `SATIS_A`, `SATIS_B`, `CHNG_A`, `CHNG_B`, and `CHNG_C`.

These columns contain recorded survey response codes. The exact meaning of each response code must be verified using the dataset documentation before interpreting the values.

When suitable variables are reshaped using `melt()`, their column names become values in a variable column, while their recorded responses are placed in a separate value column.

### 3. Which column should be treated as the identifier (`id_vars`)?

The `P_SUID` column will be used as the respondent identifier when reshaping the selected survey variables.

The initial inspection found 36,908 unique `P_SUID` values across 36,908 rows, with no missing or duplicated identifiers.

Therefore, `P_SUID` can be retained as the identifier during the reshaping operation so that each survey response remains associated with its respondent.

### **Task 1.3 – Reshape with `melt()`**

Use either of these valid Pandas forms:

```python
pd.melt(...)
```

or

```python
df.melt(...)
```

Your result should have meaningful names for the new variable and value columns, such as `income` and `count`.

In [ ]:
# Task 1.3: Reshape selected Pew Research survey variables

# Keep the original dataset unchanged
df_pew_long = df_pew.melt(
    id_vars=["P_SUID"],
    value_vars=["HAPPY", "SATIS_A", "SATIS_B"],
    var_name="survey_variable",
    value_name="response"
)

# Display the reshaped dataset
print("Original dataset shape:", df_pew.shape)
print("Reshaped dataset shape:", df_pew_long.shape)

print("\nFirst five rows of the reshaped dataset:")
display(df_pew_long.head())

# Check the survey variables included
print("\nSurvey variables included:")
print(df_pew_long["survey_variable"].unique())

### Task 1.3 – Reshape Selected Survey Variables Using `melt()`

The Pew Research dataset contains 36,908 respondents and 654 columns. For this exercise, three survey variables were selected: `HAPPY`, `SATIS_A`, and `SATIS_B`.

The `melt()` function converts these selected columns from wide format to long format. The `P_SUID` column is retained as the respondent identifier, while the selected survey variable names are stored in the `survey_variable` column and their recorded responses are stored in the `response` column.

The original dataset remains unchanged because the reshaped data is stored in a separate DataFrame named `df_pew_long`.

The original dataset has 36,908 rows and 654 columns. The reshaped dataset has 110,724 rows and three columns. The new row count is three times the original row count because three survey variables were selected for reshaping.

This transformation makes it easier to examine the selected survey variables in a consistent long-format structure. The response codes should be interpreted using the dataset documentation.

### Task 1.4 – Visualize the Reshaped Pew Research Data

The reshaped dataset contains three survey variables: `HAPPY`, `SATIS_A`, and `SATIS_B`. The `survey_variable` column identifies the original survey variable, while the `response` column contains its recorded response codes.

A bar chart will be used to compare the frequency of recorded response codes across the three selected variables. This visualization helps us examine the distribution of the recorded responses.

The response codes are treated as categories because their meanings have not yet been verified against the dataset codebook. Therefore, the chart shows response frequencies rather than assuming that larger codes represent higher levels of happiness or satisfaction.

In [ ]:
# Task 1.4: Visualize response-code frequencies

import matplotlib.pyplot as plt
import seaborn as sns

# Count responses for each survey variable
response_counts = (
    df_pew_long
    .groupby(["survey_variable", "response"], dropna=False)
    .size()
    .reset_index(name="frequency")
)

# Convert response codes to strings so they are displayed as categories
response_counts["response"] = (
    response_counts["response"].astype("string").fillna("Missing")
)

# Create the visualization
plt.figure(figsize=(12, 6))

sns.barplot(
    data=response_counts,
    x="response",
    y="frequency",
    hue="survey_variable"
)
# Display response frequencies for verification
print("Response frequencies:")
display(
    response_counts.sort_values(
        ["survey_variable", "frequency"],
        ascending=[True, False]
    )
)

plt.title("Response-Code Frequencies for Selected Pew Survey Variables")
plt.xlabel("Recorded Response Code")
plt.ylabel("Number of Responses")
plt.legend(title="Survey Variable")
plt.tight_layout()
plt.show()

### Interpretation of the Visualization

The bar chart compares the frequencies of recorded response codes for the three selected Pew Research survey variables: `HAPPY`, `SATIS_A`, and `SATIS_B`.

For `HAPPY`, response code 2 is the most frequent, with 21,525 records, followed by code 1 with 10,442 records and code 3 with 4,786 records.

For `SATIS_A`, response code 2 occurs 13,726 times, while code 3 occurs 13,030 times. Codes 1 and 4 each occur 4,615 times.

The chart helps identify the most common recorded response codes and compare their frequencies across the selected variables. These codes are treated as categories because their meanings have not yet been verified against the dataset's codebook.

### **Task 1.5 – PEW Reflection**

Answer **all** questions briefly in your own words:

1. What was the main structural problem with the original dataset?
2. What does `id_vars` mean in `melt()`?
3. What information moved from columns into rows?
4. How did the number of rows change after reshaping, and why?
5. Why is the long format easier to analyze or visualize?
6. What did your visualization reveal?

## Task 1.5 – Reflection Questions

### 1. What was the main structural problem with the original dataset?

The original workshop dataset stored income categories in separate columns, making it less convenient to analyze income as a single variable. My replacement Pew Research dataset has a different structure, with survey variables such as `HAPPY`, `SATIS_A`, and `SATIS_B` stored in separate columns. I selected three survey variables and reshaped them into long format using Pandas `melt()`.

### 2. What is the difference between wide and long data?

In wide format, different variables or categories are stored in separate columns. In long format, selected variable names are stored in one column and their corresponding values are stored in another column. Long format can make grouping, comparison, and visualization easier.

### 3. What is the purpose of the `id_vars` parameter in `melt()`?

The `id_vars` parameter identifies the columns that should remain unchanged during reshaping. In this exercise, `P_SUID` identifies each respondent and is retained in every corresponding row of the reshaped dataset.

### 4. What is the purpose of the `value_vars` parameter?

The `value_vars` parameter specifies which columns should be reshaped into rows. In this exercise, I selected `HAPPY`, `SATIS_A`, and `SATIS_B`.

### 5. How did the dataset change after reshaping?

The original dataset contained 36,908 rows and 654 columns. After reshaping the three selected survey variables, the resulting dataset contained 110,724 rows and three columns: `P_SUID`, `survey_variable`, and `response`.

The number of rows is 110,724 because each original respondent contributes up to three rows, one for each selected survey variable.

### 6. What did you learn from the visualization?

The bar chart compares the frequencies of recorded response codes for `HAPPY`, `SATIS_A`, and `SATIS_B`. It helps identify which response codes occur most frequently for each variable. The numerical codes should not be interpreted as specific answers until their meanings have been verified using the dataset's codebook.

# **Section 2 – Billboard Dataset**

## **Learning Topic: Advanced Data Tidying**

The Billboard dataset stores weekly song rankings across many week columns. You will reshape those columns into rows, clean the resulting week labels, convert ranking values to appropriate data types, handle missing values, and analyze ranking trends.

### Required concepts
- `pd.melt()` or `DataFrame.melt()`
- Pandas string methods such as:
  - `.str.replace()`
  - `.str.strip()`
  - `.str.extract()`
  - `.str.upper()`
  - `.str.lower()`
- numeric conversion
- missing-value handling
- dates / timedeltas where appropriate

## **Getting the Billboard Dataset from the Internet**

Use the **Billboard weekly ranking dataset** associated with the classic tidy-data example. The correct file contains song information plus many weekly ranking columns such as `1st week`, `2nd week`, etc. (column names can vary slightly between copies).

### Recommended method — download in your browser

1. Open this public copy of `billboard.csv`:
   `https://gist.githubusercontent.com/AdrianLl/bd94c12f2b3c3a899c4ca13d0a548966/raw/billboard.csv`
2. Save the file as **`billboard.csv`**.
3. Place it inside your project's **`data`** folder.
4. Your project should now include:

```text
project/
├── week5Lab.ipynb
└── data/
    ├── pew-raw.csv
    └── billboard.csv
```

5. Load the local file. If ordinary UTF-8 decoding fails, try `encoding="unicode_escape"`:

```python
billboard = pd.read_csv("./data/billboard.csv", encoding="unicode_escape")
```

### Optional Python-assisted download

```python
from pathlib import Path
import pandas as pd

Path("data").mkdir(exist_ok=True)

url = "https://gist.githubusercontent.com/AdrianLl/bd94c12f2b3c3a899c4ca13d0a548966/raw/billboard.csv"
billboard = pd.read_csv(url, encoding="unicode_escape")
billboard.to_csv("./data/billboard.csv", index=False)
```

### Verify before continuing

Inspect `billboard.columns`. You should see identifying information for songs/artists and a sequence of weekly ranking columns. Those repeated week columns are the columns you will later reshape with `melt()`.

> **Important:** Different online copies sometimes use slightly different column names. Do not blindly copy column names from an example. Inspect **your downloaded file** first and adapt your code accordingly.

### Billboard Dataset Provenance

This section uses the Billboard Top 100 song-ranking dataset for the year 2000 from the official `tidyverse/tidyr` repository.

The downloaded raw dataset is stored locally as `data/billboard.csv` and is preserved without manual modification. All inspection, cleaning, reshaping, and preprocessing are performed programmatically.

**Dataset source:**  
`https://raw.githubusercontent.com/tidyverse/tidyr/main/data-raw/billboard.csv`

**Official documentation:**  
`https://tidyr.tidyverse.org/reference/billboard.html`

The local file is loaded using a relative path to support reproducibility.

### **Task 2.1 – Load and Inspect Billboard Data**

1. Load the Billboard dataset.
2. If normal UTF-8 decoding fails, try `encoding="unicode_escape"`.
3. Display its shape, columns, and first rows.
4. Identify the columns that represent weekly rankings.

In [ ]:
from src.billboard_tidier import BillboardTidier


# Create the Billboard processor
billboard_processor = BillboardTidier(
    "./data/billboard.csv"
)

# Load the raw dataset
billboard = billboard_processor.load_data()

# Validate the dataset structure
id_columns, weekly_columns = (
    billboard_processor.validate_schema()
)

# Inspect the raw dataset
inspection = billboard_processor.inspect_data()


print("Dataset Shape:")
print(inspection["shape"])

print("\nIdentifier Columns:")
print(id_columns)

print("\nNumber of Weekly Ranking Columns:")
print(len(weekly_columns))

print("\nWeekly Ranking Columns:")
print(weekly_columns)

print("\nColumn Data Types:")
print(inspection["dtypes"])

print("\nNumber of Duplicate Rows:")
print(inspection["duplicate_rows"])

print("\nFirst 5 Rows:")
display(inspection["head"])

### Task 2.1 – Dataset Inspection Summary

The raw Billboard dataset contains **317 rows and 81 columns**.

Five columns describe each song:

- `year`
- `artist`
- `track`
- `time`
- `date.entered`

The remaining **76 columns**, from `wk1` through `wk76`, contain weekly Billboard chart rankings.

Initial inspection also found **0 duplicate rows**. The weekly ranking columns are numeric, while fields such as `artist`, `track`, `time`, and `date.entered` are initially stored as strings.

The dataset structure was validated programmatically using the `BillboardTidier` class before continuing with the cleaning and reshaping process.

### **Task 2.2 – Diagnose the Tidiness Problem**

Explain why having columns such as week 1, week 2, week 3, etc. makes the dataset difficult to analyze.

Identify:

- identifier columns;
- weekly measured columns; and
- the variables you want after reshaping.

### Task 2.2 – Tidiness Diagnosis

The original Billboard dataset is stored in a **wide format**. The main tidiness problem is that the week number is encoded in column names such as `wk1`, `wk2`, ..., `wk76` instead of being stored as values in a single variable.

This structure makes analysis more difficult because operations such as filtering by week, grouping rankings by week, plotting ranking changes over time, and handling missing weekly rankings would require working across many separate columns.

#### Identifier Columns

The song-level identifier columns are:

- `year`
- `artist`
- `track`
- `time`
- `date.entered`

These values describe the song and should remain associated with each weekly ranking observation.

#### Weekly Measured Columns

The measured ranking columns are:

`wk1` through `wk76`

Each of these columns stores the Billboard chart position for a particular week after the song entered the chart.

#### Desired Tidy Structure

After reshaping, the dataset should contain the original song information together with two new variables:

- `Week` – the week associated with the ranking observation;
- `Rank` – the Billboard chart position for that week.

Therefore, instead of storing one song across as many as 76 ranking columns, the tidy dataset will store **one song-week ranking observation per row**.

This long structure is easier to clean, filter, group, summarize, and visualize.

### **Task 2.3 – Reshape with `melt()`**

Reshape the weekly ranking columns into a long structure.

Your result should contain:

- the song/artist identifiers;
- a `Week`-type column; and
- a `Rank`-type column.

In [ ]:
# Reshape the Billboard dataset from wide to long format
billboard_long = billboard_processor.reshape_data()

print("Original Dataset Shape:")
print(billboard.shape)

print("\nReshaped Dataset Shape:")
print(billboard_long.shape)

print("\nColumns After Reshaping:")
print(billboard_long.columns.tolist())

print("\nFirst 10 Rows of the Reshaped Dataset:")
display(billboard_long.head(10))

In [ ]:
expected_rows = len(billboard) * len(weekly_columns)

print("\nExpected Number of Rows:")
print(expected_rows)

print("\nActual Number of Rows:")
print(len(billboard_long))

print("\nReshape Validation:")
print(len(billboard_long) == expected_rows)

### Task 2.3 – Reshaping Summary

The Billboard dataset was reshaped from wide format to long/tidy format using Pandas `melt()`.

The five song-level identifier columns were preserved:

- `year`
- `artist`
- `track`
- `time`
- `date.entered`

The 76 weekly ranking columns (`wk1` through `wk76`) were converted into two variables:

- `Week` – identifies the weekly ranking period;
- `Rank` – stores the Billboard chart position.

The original dataset contained **317 rows and 81 columns**. After reshaping, each song can contribute up to 76 song-week observations.

The expected number of rows is:

`317 × 76 = 24,092`

This long format is more suitable for cleaning, filtering, grouping, date calculations, and ranking-trend visualization.

### **Task 2.4 – Clean the Week Labels with String Operations**

Use Pandas string manipulation to clean the week labels.

Demonstrate **at least three** relevant string operations. Examples include:

```python
.str.replace(...)
.str.strip()
.str.extract(...)
.str.lower()
.str.upper()
```

Convert the extracted week number to a numeric type after cleaning.

In [ ]:
# Clean Billboard week labels using the OOP preprocessing module
billboard_long = billboard_processor.clean_week_labels()

print("Week Labels Before and After Cleaning:")
display(
    billboard_long[
        ["Week_Original", "Week", "Week_Number"]
    ].drop_duplicates().head(15)
)

print("\nWeek_Number Data Type:")
print(billboard_long["Week_Number"].dtype)

print("\nMinimum Week Number:")
print(billboard_long["Week_Number"].min())

print("\nMaximum Week Number:")
print(billboard_long["Week_Number"].max())

print("\nMissing Week Numbers:")
print(billboard_long["Week_Number"].isna().sum())

print("\nNumber of Unique Week Numbers:")
print(billboard_long["Week_Number"].nunique())

### Task 2.4 – Week Label Cleaning Summary

The original weekly labels were stored as strings such as `wk1`, `wk2`, ..., `wk76`.

To prepare these labels for analysis, four Pandas string operations were applied:

1. `.str.strip()` removed leading and trailing whitespace.
2. `.str.lower()` standardized the text to lowercase.
3. `.str.replace()` replaced the `wk` prefix with the clearer label `week`.
4. `.str.extract()` extracted the numeric portion of each label.

The extracted week value was converted to a numeric data type using `pd.to_numeric()`.

Validation confirmed that `Week_Number` is an integer, contains **76 unique weeks**, ranges correctly from **1 to 76**, and contains **no missing values**. Numeric week values are important for correct sorting, date calculations, and ranking-trend visualization.

### **Task 2.5 – Clean Ranking Values and Missing Data**

1. Convert ranking values to numeric using an appropriate method such as `pd.to_numeric()`.
2. Identify missing rankings.
3. Decide whether missing ranking rows should be dropped, retained, or treated another way.
4. Explain your decision.

In [ ]:
# Clean ranking values and assess missing rankings
billboard_clean, ranking_summary = (
    billboard_processor.clean_rankings()
)

print("Billboard Ranking Cleaning Summary:")
display(
    pd.DataFrame(
        [ranking_summary]
    ).T.rename(columns={0: "Value"})
)

print("\nRank Data Type:")
print(billboard_clean["Rank"].dtype)

print("\nRemaining Missing Rankings:")
print(billboard_clean["Rank"].isna().sum())

print("\nFirst 10 Valid Ranking Observations:")
display(
    billboard_clean[
        [
            "artist",
            "track",
            "Week",
            "Week_Number",
            "Rank"
        ]
    ].head(10)
)

### Task 2.5 – Ranking and Missing-Value Treatment

The `Rank` column was converted to numeric values using `pd.to_numeric()`.

After reshaping, the dataset contained **24,092 possible song-week observations**. Of these, **18,785 rankings were missing**, representing approximately **77.97%** of the reshaped rows.

The large number of missing values is expected because most songs did not remain on the Billboard chart for all 76 possible weeks.

Missing ranking observations were **not imputed**. A missing value indicates that no chart position was recorded for that song during that week. Replacing these values with zero, a mean, a median, or another artificial value would create Billboard rankings that did not actually occur.

Therefore, rows without a recorded ranking were removed from the analysis-ready dataset while the original and long-format data remained preserved.

After treatment:

- **5,307 valid song-week ranking observations remained**
- **0 ranking values were missing**
- valid rankings ranged from **1 to 100**
- **0 rankings were found outside the valid 1–100 range**

This produces an analysis-ready dataset while preserving the original raw data separately.

### **Task 2.6 – Optional Date Enhancement**

If the dataset contains a song's entry date, calculate the approximate chart date for each weekly observation using a `Timedelta`.

Document the equation you use and check for off-by-one-week errors.

In [ ]:
# Add entry dates and calculated weekly chart dates
billboard_clean = billboard_processor.add_chart_dates()

print("Entry_Date Data Type:")
print(billboard_clean["Entry_Date"].dtype)

print("\nChart_Date Data Type:")
print(billboard_clean["Chart_Date"].dtype)

print("\nMissing Entry Dates:")
print(billboard_clean["Entry_Date"].isna().sum())

print("\nMissing Chart Dates:")
print(billboard_clean["Chart_Date"].isna().sum())


# Verify the week-date calculation using one song
date_check = (
    billboard_clean[
        (billboard_clean["artist"] == "3 Doors Down")
        & (billboard_clean["track"] == "Kryptonite")
    ][
        [
            "artist",
            "track",
            "date.entered",
            "Week_Number",
            "Rank",
            "Entry_Date",
            "Chart_Date"
        ]
    ]
    .sort_values("Week_Number")
    .head(5)
)

print("\nOff-by-One Date Validation:")
display(date_check)

### Task 2.6 – Date Enhancement Summary

The original `date.entered` field was converted from a string to a Pandas datetime value using `pd.to_datetime()`.

A corresponding `Chart_Date` was then calculated for each weekly ranking using the following formula:

**Chart Date = Entry Date + (Week Number - 1) × 7 days**

The subtraction of 1 is important because `Week_Number = 1` represents the song's entry week. Therefore, Week 1 uses the original entry date, while each following week adds another 7 days.

The subtraction of 1 is important because `Week_Number = 1` represents the song's entry week and should therefore use the original entry date.

The calculation was validated using **3 Doors Down – Kryptonite**:

- Week 1: 2000-04-08
- Week 2: 2000-04-15
- Week 3: 2000-04-22
- Week 4: 2000-04-29
- Week 5: 2000-05-06

There were **0 missing entry dates** and **0 missing calculated chart dates**, confirming that the date transformation was successful.

### **Task 2.7 – Billboard Visualization and Trend Analysis**

Create at least **one visualization** showing a meaningful Billboard trend.

Examples:

- ranking over time for a selected song;
- comparison of ranking trajectories for multiple songs;
- distribution of chart ranks by week.

Remember: on Billboard, a **smaller rank number is better**.

In [ ]:
# Plot the weekly Billboard ranking trend
kryptonite_trend = billboard_processor.plot_song_trend(
    artist="3 Doors Down",
    track="Kryptonite"
)

# Calculate trend statistics
best_rank = kryptonite_trend["Rank"].min()

peak_row = kryptonite_trend.loc[
    kryptonite_trend["Rank"].idxmin()
]

weeks_charted = len(kryptonite_trend)

first_rank = kryptonite_trend.iloc[0]["Rank"]
last_rank = kryptonite_trend.iloc[-1]["Rank"]

print("Trend Summary:")
print("Song: 3 Doors Down - Kryptonite")
print(f"Number of Ranked Weeks: {weeks_charted}")
print(f"First Recorded Rank: {first_rank}")
print(f"Best Rank: {best_rank}")
print(f"Week of Best Rank: {peak_row['Week_Number']}")
print(f"Date of Best Rank: {peak_row['Chart_Date'].date()}")
print(f"Last Recorded Rank: {last_rank}")

In [ ]:
# I also want to show a small trend Table
display(
    kryptonite_trend[
        [
            "Week_Number",
            "Chart_Date",
            "Rank"
        ]
    ].head(15)
)

### Task 2.7 – Billboard Trend Analysis

The chart shows the weekly Billboard ranking trend for **3 Doors Down – Kryptonite**.

The song entered the dataset at **rank 81** and improved steadily over the following weeks. Its best recorded position was **rank 3**, which it first reached in **Week 32** on **November 11, 2000**.

The song remained close to the top of the chart for several weeks before its ranking gradually declined. Its final recorded position in this dataset was **rank 49**, after a total of **53 ranked weeks**.

The y-axis is displayed in reverse order because a smaller Billboard ranking number represents better performance. Therefore, movement upward on the graph represents an improvement in chart position.

This trend demonstrates why converting the original wide weekly columns into long format is useful: the cleaned data can be ordered by week and plotted as a continuous ranking history.

In [ ]:
print("FINAL BILLBOARD VALIDATION")
print("-" * 40)

print(f"Raw dataset shape: {billboard.shape}")
print(f"Long-format shape: {billboard_long.shape}")
print(f"Analysis-ready shape: {billboard_clean.shape}")

print(f"\nDuplicate rows in raw data: {billboard.duplicated().sum()}")
print(f"Missing ranks after cleaning: {billboard_clean['Rank'].isna().sum()}")
print(f"Missing week numbers: {billboard_clean['Week_Number'].isna().sum()}")
print(f"Missing chart dates: {billboard_clean['Chart_Date'].isna().sum()}")

print(
    f"Week range: "
    f"{billboard_clean['Week_Number'].min()} "
    f"to {billboard_clean['Week_Number'].max()}"
)

print(
    f"Rank range: "
    f"{billboard_clean['Rank'].min()} "
    f"to {billboard_clean['Rank'].max()}"
)

print(
    "Ranks outside 1-100:",
    (~billboard_clean["Rank"].between(1, 100)).sum()
)

### **Task 2.8 – Billboard Reflection**

Answer all questions:

1. Why was `melt()` appropriate for the weekly ranking columns?
2. Which string operations did you use and what did each accomplish?
3. Why should the cleaned week number be numeric?
4. How did you handle missing ranking values, and why?
5. What trend did your visualization reveal?
6. What data-quality issue could lead to an incorrect ranking trend?
7. How is this Billboard transformation more complex than the PEW transformation?

### **Task 2.8 – Billboard Reflection**

Answer all questions:

**1. Why was `melt()` appropriate for the weekly ranking columns?**

`melt()` was appropriate because the weekly rankings were stored across many separate columns such as `wk1`, `wk2`, ..., `wk76`. These columns represent repeated measurements of the same variable over time. Using `melt()` converted the dataset from wide format to long format, where the week is stored in one column and the ranking is stored in another. This makes the data easier to clean, filter, analyze, and visualize.

**2. Which string operations did you use and what did each accomplish?**

Four Pandas string operations were used:

- `.str.strip()` removed any leading or trailing whitespace.
- `.str.lower()` standardized the week labels to lowercase.
- `.str.replace()` replaced the `wk` prefix with the clearer label `week`.
- `.str.extract()` extracted the numeric part of the week label.

For example, `wk10` was transformed into `week 10`, and the number `10` was stored separately in `Week_Number`.

**3. Why should the cleaned week number be numeric?**

The cleaned week number should be numeric so that the weeks can be sorted, compared, calculated, and plotted correctly. Numeric week values are also required for creating the weekly chart dates. If the weeks remained as strings, values such as `week 10` could be ordered incorrectly relative to `week 2`.

**4. How did you handle missing ranking values, and why?**

After reshaping, there were **18,785 missing ranking values out of 24,092 possible song-week observations**, which was approximately **77.97%** of the long-format dataset.

The missing rankings were not imputed because a missing value represents a week in which no Billboard ranking was recorded for that song. Filling these values with zero, the mean, median, or another value would create an artificial chart ranking that did not actually occur.

Therefore, rows with missing rankings were removed from the analysis-ready dataset. This left **5,307 valid ranking observations**, with rankings ranging from **1 to 100**.

**5. What trend did your visualization reveal?**

The visualization for **3 Doors Down – Kryptonite** showed that the song started at **rank 81** and gradually improved over time. It reached its best position of **rank 3 in Week 32**, corresponding to **November 11, 2000**.

The song remained close to the top of the Billboard chart for several weeks and then gradually declined. Its final recorded rank was **49**, and it had a total of **53 ranked weeks**.

**6. What data-quality issue could lead to an incorrect ranking trend?**

One data-quality issue that could lead to an incorrect ranking trend is treating missing ranking values as actual chart positions. For example, replacing a missing value with zero would incorrectly make an unranked week appear better than rank 1.

Incorrectly sorting week labels as strings could also produce the wrong chronological order. Converting `Week_Number` to numeric form and excluding missing rankings helps prevent these problems.

**7. How is this Billboard transformation more complex than the PEW transformation?**

The Billboard transformation is more complex because it required more than simply reshaping the dataset. In addition to using `melt()`, the week labels had to be cleaned using string operations, the week numbers had to be converted to numeric values, ranking values had to be validated, missing rankings required a justified treatment decision, and entry dates were converted into weekly chart dates.

The PEW transformation mainly required converting category columns from wide format to long format, while the Billboard dataset required several additional cleaning and preprocessing steps before the data was ready for trend analysis.

# **Section 3 – Data Cleaning (Cars Dataset)**

## **Learning Topic: Data Cleaning and Missing Data Handling**

Data cleaning includes identifying invalid, irrelevant, duplicated, inconsistent, or missing data and deciding how to treat it.

In this section you will work with the Auto MPG / Cars dataset and compare several missing-data strategies.

### Required functions and concepts

- `isnull()` / `isna()`
- `notnull()` / `notna()`
- `dropna()`
- `fillna()`
- `mean()`
- `median()`
- `mode()`
- duplicate detection/removal
- numeric conversion
- `SimpleImputer`

## **Getting the Cars (Auto MPG) Dataset from the Internet**

Use the **Auto MPG** dataset from the UCI Machine Learning Repository. UCI describes it as a regression dataset concerning city-cycle fuel consumption. It contains variables such as MPG, cylinders, displacement, horsepower, weight, acceleration, model year, origin, and car name.

### Recommended source

Official UCI dataset page:

`https://archive.ics.uci.edu/dataset/9/auto+mpg`

The UCI page provides the dataset files and documentation. The raw file is named **`auto-mpg.data`** rather than `cars.csv`.

### Recommended student workflow

1. Open the UCI page above.
2. Download the Auto MPG dataset.
3. Extract the downloaded archive if necessary.
4. Locate **`auto-mpg.data`**.
5. Place it in your project's `data/` folder.
6. Keep the original raw file rather than editing it manually.

Your project can look like:

```text
project/
├── week5Lab.ipynb
└── data/
    └── auto-mpg.data
```

### Loading the UCI raw file with Pandas

Because this file is whitespace-delimited rather than a normal comma-separated CSV, supply the column names when reading it:

```python
column_names = [
    "MPG", "Cylinders", "Displacement", "Horsepower",
    "Weight", "Acceleration", "ModelYear", "Origin", "CarName"
]

cars = pd.read_csv(
    "./data/auto-mpg.data",
    sep=r"\s+",
    names=column_names,
    na_values="?",
    quotechar='"'
)
```

The `na_values="?"` argument is important because the UCI dataset uses `?` for missing horsepower values.

### Optional: create a local CSV working copy

After correctly loading the original UCI file, you may save a CSV copy:

```python
cars.to_csv("./data/cars.csv", index=False)
```

You can then reload it later with:

```python
cars = pd.read_csv("./data/cars.csv")
```

### Verify before continuing

Check:

```python
cars.shape
cars.head()
cars.dtypes
cars.isnull().sum()
```

The UCI version contains **398 instances**, and `Horsepower` contains missing values. If your results are substantially different, verify your download and loading code.

> **Good practice:** Preserve the original raw file. If you create `cars.csv`, treat it as a derived working copy rather than silently replacing the source data.

### **Task 3.1 – Load and Inspect the Cars Dataset**

1. Load the Cars dataset from your data folder.
2. Inspect shape, columns, data types, and sample rows.
3. Determine whether any row contains metadata or data-type labels rather than an actual car observation.
4. Remove irrelevant rows **only if justified by inspection**.

In [ ]:
# TODO: Load the Cars dataset.
# TODO: Inspect shape, columns, dtypes, head/tail.
# TODO: Remove any clearly irrelevant/non-data rows if present.

### **Task 3.2 – Assess Missing and Invalid Values**

1. Count missing values in every column.
2. Calculate the percentage of missing values.
3. Use `notnull()` or `notna()` at least once to confirm valid observations.
4. Check for values that should be numeric but are stored as text.
5. Convert columns to appropriate data types where needed.

In [ ]:
# TODO: Count and calculate percentages of missing values.
# TODO: Demonstrate notnull()/notna().
# TODO: Convert appropriate columns to numeric.

### **Task 3.3 – Check Duplicates**

Identify duplicate records.

- Report the number of duplicates.
- Remove duplicates if present.
- If there are none, state that clearly.

In [ ]:
# TODO: Detect duplicate rows.
# TODO: Remove duplicates if appropriate.

### **Task 3.4 – Compare Missing-Value Strategies**

Create copies of the data so you can compare approaches.

Demonstrate:

1. dropping rows with missing values using `dropna()`;
2. dropping columns with missing values and explaining why this may be too aggressive;
3. filling a numeric variable with its **mean**;
4. filling a numeric variable with its **median**;
5. using **mode** where appropriate for a categorical/discrete variable.

Do not automatically assume one strategy is best.

In [ ]:
# TODO: Demonstrate dropna() on rows.
# TODO: Demonstrate the effect of dropna(axis=1).
# TODO: Demonstrate mean, median, and mode imputation on suitable copies/columns.

### **Task 3.5 – Use Visualization to Support an Imputation Decision**

Choose a numeric feature such as `MPG`.

1. Plot its distribution.
2. Describe whether it appears symmetric or skewed.
3. Use that evidence to justify whether **mean or median** is a better simple imputation choice.

In [ ]:
# TODO: Plot the distribution of a numeric feature such as MPG.
# TODO: Calculate its mean and median.

### **Task 3.6 – Apply `SimpleImputer`**

Use Scikit-Learn's `SimpleImputer` on at least one appropriate feature.

Remember the two main stages:

1. `fit` – learn the replacement statistic from the data;
2. `transform` – replace missing values using that statistic.

You may use `fit_transform()` when appropriate, but explain what it combines.

In [ ]:
# TODO: Create a SimpleImputer.
# TODO: Fit it to one or more appropriate columns.
# TODO: Transform the data.
# TODO: Verify that the selected missing values were handled.

### **Task 3.7 – Validate the Cleaned Cars Dataset**

Compare the dataset **before and after cleaning**.

Report:

- shape;
- missing-value counts;
- duplicate counts;
- descriptive statistics for selected numeric columns.

Create at least **one visualization** supporting your comparison.

In [ ]:
# TODO: Compare before/after statistics and data quality.
# TODO: Create a supporting visualization.

### **Task 3.8 – Cars Reflection**

Answer all questions:

1. What were the main data-quality problems?
2. How many missing values did you find?
3. What happened when you dropped rows containing missing values?
4. Why might dropping entire columns be inappropriate?
5. When is mean imputation reasonable?
6. When might median be safer than mean?
7. Where would mode imputation be appropriate?
8. How did `SimpleImputer` differ from manually calling `fillna()`?
9. Did cleaning materially change the dataset statistics?
10. Which cleaning decisions would you carry forward into an ML pipeline, and why?

> **TODO – Cars observations and reflection:**  
> Write your answers here.

# **Section 4 – Outlier Detection (Diabetes Dataset)**

## **Learning Topic: Outlier Detection and Treatment**

An **outlier** is an observation that differs substantially from most of the data. Outliers can result from measurement errors, data-entry problems, unusual but legitimate cases, or genuinely rare events.

Outliers matter because they can affect:

- means and standard deviations;
- correlations;
- regression coefficients;
- distance-based algorithms;
- scaling and normalization;
- model evaluation.

**Important:** Detecting an outlier does **not** automatically mean it should be deleted. You must justify treatment in the context of the data.

## **Getting the Diabetes Dataset from the Internet**

For this section, use the **Pima Indians Diabetes** CSV commonly used in introductory machine-learning exercises. The expected version has **768 observations** and these nine columns:

`Pregnancies`, `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, `BMI`, `DiabetesPedigreeFunction`, `Age`, and `Outcome`.

### Recommended method — download in your browser

A public CSV copy is available here:

`https://raw.githubusercontent.com/npradaschnor/Pima-Indians-Diabetes-Dataset/master/diabetes.csv`

1. Open the address above in a browser.
2. Save the file as **`diabetes.csv`**.
3. Place it in your project's **`data`** folder.
4. Load the local copy:

```python
diabetes = pd.read_csv("./data/diabetes.csv")
```

### Optional Python-assisted download

```python
from pathlib import Path
import pandas as pd

Path("data").mkdir(exist_ok=True)

url = "https://raw.githubusercontent.com/npradaschnor/Pima-Indians-Diabetes-Dataset/master/diabetes.csv"
diabetes = pd.read_csv(url)
diabetes.to_csv("./data/diabetes.csv", index=False)
```

### Verify before continuing

Run:

```python
diabetes.shape
diabetes.head()
diabetes.columns
```

For the expected file, the shape should be `(768, 9)` and the columns should match the list above.

### Important data-quality warning

In this dataset, some physiological variables contain **zero values** that may represent unavailable or invalid measurements rather than plausible measurements. Do not automatically treat every zero as an outlier or delete it. Inspect the variable meaning first, document your decision, and keep the focus of this section on comparing the required outlier-detection methods.

> **Dataset-name warning:** UCI also hosts a dataset simply called **Diabetes**, but it is a different time-series dataset. For this particular lab, use the nine-column Pima-style CSV described above so your work matches the assignment.

## Section 4 - CDC Diabetes Health Indicators

### 4.1 Data Acquisition

For this section, I used the CDC Diabetes Health Indicators dataset from the UCI Machine Learning Repository.

Instead of manually loading a local CSV file, I accessed the dataset using the `ucimlrepo` Python package. This makes the data source easier to reproduce because another user can retrieve the same dataset directly from UCI.

The dataset contains health, lifestyle, and demographic information that can be used to study factors related to diabetes.

In [ ]:
from ucimlrepo import fetch_ucirepo

# Fetch the CDC Diabetes Health Indicators dataset from UCI
cdc_diabetes = fetch_ucirepo(id=891)

# Separate features and target
X = cdc_diabetes.data.features
y = cdc_diabetes.data.targets

# Combine them into one DataFrame
diabetes = pd.concat([X, y], axis=1)

print("Dataset loaded successfully.")
print("Shape:", diabetes.shape)

diabetes.head()

print("Shape:")
print(diabetes.shape)

print("\nColumns:")
print(diabetes.columns)

print("\nData types:")
print(diabetes.dtypes)

print("\nFirst 5 rows:")
display(diabetes.head())

print("\nDescriptive statistics:")
display(diabetes.describe())

In [ ]:
print("Dataset Name:")
print(cdc_diabetes.metadata.name)

print("\nNumber of Instances:")
print(cdc_diabetes.metadata.num_instances)

print("\nNumber of Features:")
print(cdc_diabetes.metadata.num_features)

print("\nVariable Information:")
display(cdc_diabetes.variables)

### Understanding the Dataset

Before cleaning the dataset, I first checked its metadata and variables.

This helped me understand what each column represents and which columns are continuous values, categorical values, or binary indicators.

This is important because different types of variables should not always be cleaned in the same way.

In [ ]:
print("Missing values:")
display(diabetes.isnull().sum())

print("Duplicate rows:")
print(diabetes.duplicated().sum())

print("Data types:")
display(diabetes.dtypes)

print("Shape:")
print(diabetes.shape)

### Data Quality Check

Before detecting outliers, I checked the dataset for missing values, duplicate rows, and data types.

These checks are useful because poor-quality data can affect later analysis. For example, duplicate rows can give some observations more influence than they should have, while missing values may require removal or imputation.

In [ ]:
diabetes["Diabetes_binary"].value_counts()

In [ ]:
diabetes["Diabetes_binary"].value_counts(normalize=True) * 100

### Diabetes Class Distribution

I checked the target variable to understand how many people in the dataset were classified as diabetic and non-diabetic.

This is useful because a large difference between the two groups can affect how the dataset is interpreted and can become important later if the data is used for machine learning.

In [ ]:
diabetes["Health_Risk_Count"] = (
    diabetes["HighBP"]
    + diabetes["HighChol"]
    + diabetes["Smoker"]
    + diabetes["Stroke"]
    + diabetes["HeartDiseaseorAttack"]
)

### Feature Engineering

I created a new feature called `Health_Risk_Count`.

This feature combines several binary health-risk indicators into one value. A higher value means that a person has more of the selected health risk factors.

I created this feature to make it easier to compare overall health risk between people instead of looking at every indicator separately.

### **Task 4.2 – Box Plot Method**

Create box plots for selected numeric features.

Use the plots to:

- identify observations outside the whiskers;
- compare which variables appear to contain more extreme values;
- explain what the box, median, quartiles, and whiskers represent.

In [ ]:
sns.boxplot(
    data=diabetes,
    x="Diabetes_binary",
    y="BMI"
)

plt.title("BMI Distribution by Diabetes Status")
plt.xlabel("Diabetes Status")
plt.ylabel("BMI")
plt.show()

### **Task 4.3 – Scatter Plot Method**

Create at least one scatter plot using two meaningful numeric features.

Use the plot to identify observations that appear isolated from the main cluster of data.

In [ ]:
sns.scatterplot(
    data=diabetes.sample(5000, random_state=42),
    x="BMI",
    y="PhysHlth",
    hue="Diabetes_binary",
    alpha=0.5
)

plt.title("BMI vs Physical Health by Diabetes Status")
plt.xlabel("BMI")
plt.ylabel("Poor Physical Health Days")
plt.show()

#### Scatter Plot Interpretation

The scatter plot compares BMI with the number of poor physical health days reported in the last 30 days.

Most observations are concentrated between a BMI of approximately 20 and 45. I can also see a few unusually high BMI values above 60, including some values above 80.

The horizontal lines appear because `PhysHlth` contains whole-number values from 0 to 30 rather than continuous decimal values.

Both diabetic and non-diabetic individuals appear throughout the graph, although diabetic observations can also be seen among people with higher BMI and more poor physical health days.

This visualization helped me identify possible extreme BMI values that should be examined further using statistical outlier detection methods.

### 4.4 Outlier Detection Using Z-Score

The Z-score measures how far a value is from the mean in terms of standard deviations.

A Z-score close to 0 means the value is close to the average. A large positive or negative Z-score means the value is farther away from the average.

For this analysis, I will consider BMI values with an absolute Z-score greater than 3 as potential outliers.

In [ ]:
# Calculate the mean and standard deviation of BMI
bmi_mean = diabetes["BMI"].mean()
bmi_std = diabetes["BMI"].std()

# Calculate Z-score for each BMI value
diabetes["BMI_Z"] = (
    diabetes["BMI"] - bmi_mean
) / bmi_std

# Find BMI values more than 3 standard deviations from the mean
z_outliers = diabetes[
    diabetes["BMI_Z"].abs() > 3
]

print("BMI Mean:", bmi_mean)
print("BMI Standard Deviation:", bmi_std)
print("Number of Z-score outliers:", len(z_outliers))

display(
    z_outliers[
        ["BMI", "BMI_Z", "Diabetes_binary"]
    ].head(10)
)

#### Z-Score Interpretation

The average BMI in the dataset is approximately 28.38, with a standard deviation of about 6.61.

Using the rule `|Z-score| > 3`, I identified 2,963 BMI values as potential outliers.

For example, BMI values such as 49, 50, 55, and 59 have Z-scores above 3, which means they are much higher than the average BMI in the dataset.

At this stage, I am only identifying these values as potential outliers. I am not removing them yet because I want to compare the Z-score results with another method, the Interquartile Range (IQR).

### 4.5 Outlier Detection Using IQR

The Interquartile Range (IQR) method identifies outliers using the middle 50% of the data.

First, I calculate the first quartile (Q1) and third quartile (Q3). The difference between them is called the IQR.

Values below `Q1 - 1.5 × IQR` or above `Q3 + 1.5 × IQR` are considered potential outliers.

I will apply this method to BMI and compare the results with the Z-score method.

In [ ]:
# Calculate Q1 and Q3 for BMI
Q1 = diabetes["BMI"].quantile(0.25)
Q3 = diabetes["BMI"].quantile(0.75)

# Calculate IQR
IQR = Q3 - Q1

# Calculate lower and upper limits
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Find BMI outliers
iqr_outliers = diabetes[
    (diabetes["BMI"] < lower_bound) |
    (diabetes["BMI"] > upper_bound)
]

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)
print("Number of IQR outliers:", len(iqr_outliers))

display(
    iqr_outliers[
        ["BMI", "Diabetes_binary"]
    ].head(10)
)

#### IQR Interpretation

For BMI, the first quartile (Q1) is 24 and the third quartile (Q3) is 31. This gives an IQR of 7.

Using the IQR rule, BMI values below 13.5 or above 41.5 were identified as potential outliers.

This method identified 9,847 possible outliers, which is much higher than the 2,963 outliers identified using the Z-score method.

These values should not automatically be considered errors because very high or low BMI values can still represent real people. The purpose of this step is to identify unusual observations before deciding how they should be handled.

### 4.6 Comparing Z-Score and IQR Outliers

I used both Z-score and IQR to detect unusual BMI values.

The two methods may identify different numbers of outliers because they use different approaches. Z-score uses the mean and standard deviation, while IQR uses the middle 50% of the data.

Comparing both methods helps me make a more informed decision before removing or modifying any observations.

In [ ]:
print("Z-score outliers:", len(z_outliers))
print("IQR outliers:", len(iqr_outliers))

# Find rows identified as outliers by both methods
common_outliers = z_outliers.index.intersection(iqr_outliers.index)

print("Outliers identified by both methods:", len(common_outliers))

#### Comparison of Z-Score and IQR

The Z-score method identified 2,963 BMI outliers, while the IQR method identified 9,847.

All 2,963 Z-score outliers were also identified by the IQR method. This shows that the Z-score method detected the more extreme BMI values, while the IQR method also detected less extreme values.

Because BMI values above the normal range can still represent real people, I do not want to remove every value identified by IQR. I will use the more conservative Z-score method for outlier treatment so that fewer valid observations are removed.

### 4.7 Outlier Treatment

After comparing the two methods, I decided to use the Z-score method for treatment.

The IQR method identified a much larger number of BMI values as outliers, but some of these values may still be realistic. Therefore, I used the more conservative rule of `|Z-score| > 3` to remove only the most extreme BMI observations.

I created a new cleaned DataFrame so that the original dataset remains unchanged.

In [ ]:
# Create a copy of the original dataset
diabetes_clean = diabetes.copy()

# Remove only extreme BMI values using the Z-score rule
diabetes_clean = diabetes_clean[
    diabetes_clean["BMI_Z"].abs() <= 3
].copy()

print("Rows before treatment:", len(diabetes))
print("Rows after treatment:", len(diabetes_clean))
print("Rows removed:", len(diabetes) - len(diabetes_clean))

print("\nBMI before treatment:")
print(diabetes["BMI"].describe())

print("\nBMI after treatment:")
print(diabetes_clean["BMI"].describe())

#### Outlier Treatment Interpretation

Before treatment, the dataset contained 253,680 rows. After removing BMI observations with an absolute Z-score greater than 3, 250,717 rows remained.

A total of 2,963 rows were removed.

The average BMI changed only slightly, from approximately 28.38 to 28.02. However, the standard deviation decreased from approximately 6.61 to 5.61.

This shows that removing the extreme BMI values reduced the spread of the data while keeping the overall average relatively similar. I used a conservative treatment approach because unusual BMI values may still represent real people and should not automatically be considered data errors.

### 4.8 Before and After Outlier Treatment

To understand the effect of the outlier treatment, I compared the BMI distribution before and after removing the extreme Z-score outliers.

The purpose of this comparison is to check whether the treatment reduced extreme values without significantly changing the overall pattern of the data.

In [ ]:
# BMI before outlier treatment
plt.figure(figsize=(8, 4))

sns.boxplot(x=diabetes["BMI"])

plt.title("BMI Before Outlier Treatment")
plt.xlabel("BMI")

plt.show()

In [ ]:
# BMI after outlier treatment
plt.figure(figsize=(8, 4))

sns.boxplot(x=diabetes_clean["BMI"])

plt.title("BMI After Outlier Treatment")
plt.xlabel("BMI")

plt.show()

In [ ]:
# BMI distribution before treatment
plt.figure(figsize=(8, 4))

sns.histplot(
    diabetes["BMI"],
    bins=30,
    kde=True
)

plt.title("BMI Distribution Before Treatment")
plt.xlabel("BMI")
plt.ylabel("Frequency")

plt.show()

In [ ]:
# BMI distribution after treatment
plt.figure(figsize=(8, 4))

sns.histplot(
    diabetes_clean["BMI"],
    bins=30,
    kde=True
)

plt.title("BMI Distribution After Treatment")
plt.xlabel("BMI")
plt.ylabel("Frequency")

plt.show()

#### Before and After Treatment Interpretation

After removing the extreme BMI values using the Z-score method, the overall shape of the BMI distribution remained similar.

Most observations are still concentrated around the middle BMI range, while the very extreme values on the right side were reduced.

The distribution is still slightly right-skewed, which means some higher BMI values remain. These values were kept because they were not extreme enough to meet the Z-score threshold.

This suggests that the treatment reduced extreme observations without greatly changing the main pattern of the dataset.

### 4.9 Feature Engineering

Feature engineering means creating a new useful variable from existing variables.

For this dataset, I created a feature called `Health_Risk_Count`.

This feature adds several binary health-risk indicators together. A higher value means that a person has more of the selected health risk factors.

This can make the dataset easier to analyze because it summarizes several related health conditions into one feature.

In [ ]:
diabetes_clean["Health_Risk_Count"] = (
    diabetes_clean["HighBP"]
    + diabetes_clean["HighChol"]
    + diabetes_clean["Smoker"]
    + diabetes_clean["Stroke"]
    + diabetes_clean["HeartDiseaseorAttack"]
)

print(
    diabetes_clean["Health_Risk_Count"]
    .value_counts()
    .sort_index()
)

In [ ]:
plt.figure(figsize=(8, 4))

sns.countplot(
    data=diabetes_clean,
    x="Health_Risk_Count"
)

plt.title("Distribution of Health Risk Count")
plt.xlabel("Number of Health Risk Factors")
plt.ylabel("Number of People")

plt.show()

#### Health Risk Count Interpretation

The graph shows how many selected health risk factors each person has.

Most people in the dataset have between 0 and 2 risk factors. The largest group has 1 risk factor, while fewer people have 3 or more.

Only a small number of people have all 5 selected risk factors.

This new feature makes it easier to summarize several health conditions into one value. However, it does not mean that every risk factor has exactly the same medical importance. It is mainly used here as a simple feature for exploratory analysis.

### 4.10 Relationship Between Health Risk Count and Diabetes

After creating the `Health_Risk_Count` feature, I wanted to check whether diabetes was more common among people with a larger number of the selected health risk factors.

Because `Diabetes_binary` contains 0 and 1 values, the average of this column within each risk-count group represents the proportion of people with diabetes.

In [ ]:
# Calculate diabetes percentage for each Health Risk Count
diabetes_rate = (
    diabetes_clean
    .groupby("Health_Risk_Count")["Diabetes_binary"]
    .mean()
    * 100
)

print("Diabetes percentage by number of health risk factors:")
display(diabetes_rate)

In [ ]:
plt.figure(figsize=(8, 4))

diabetes_rate.plot(kind="bar")

plt.title("Diabetes Percentage by Health Risk Count")
plt.xlabel("Number of Health Risk Factors")
plt.ylabel("Diabetes Percentage (%)")
plt.xticks(rotation=0)

plt.show()

#### Health Risk Count and Diabetes Interpretation

The graph shows a clear increase in the percentage of people with diabetes as the number of selected health risk factors increases.

People with no selected risk factors have the lowest diabetes percentage, at approximately 3%. The percentage increases as more risk factors are present and reaches approximately 46% for people with all 5 selected risk factors.

This suggests that the combined `Health_Risk_Count` feature is useful for identifying groups with a higher occurrence of diabetes.

However, this graph only shows an association. It does not prove that these risk factors directly cause diabetes. Other health, lifestyle, and demographic factors may also influence the result.

### 4.11 Final Reflection

In this section, I worked with the CDC Diabetes Health Indicators dataset and practiced several data preprocessing and analysis techniques.

I first checked the dataset structure, missing values, duplicate rows, and data types. I then used visual methods such as box plots and scatter plots to explore possible unusual values.

For BMI outlier detection, I applied both the Z-score and IQR methods. The two methods identified different numbers of outliers because they use different rules. I chose the more conservative Z-score approach for treatment so that fewer potentially valid observations were removed.

After treatment, the average BMI remained similar, while the spread of the data decreased. This showed that the extreme values were reduced without greatly changing the overall dataset.

I also created a new feature called `Health_Risk_Count` by combining several health-risk indicators. The analysis showed that the percentage of people with diabetes increased as the number of selected risk factors increased.

From this section, I learned that data cleaning and outlier detection require judgment. A value should not be removed only because it looks unusual. It is important to understand the data, compare different methods, and explain why a cleaning decision was made.

#### 1. What is an outlier?

An outlier is a data value that is very different from most of the other observations in a dataset. It may be unusually high or unusually low compared with the general pattern of the data.

#### 2. What are at least three possible causes of outliers?

Some possible causes of outliers are:

- Data entry mistakes, such as typing the wrong value.
- Measurement or sensor errors.
- A real but uncommon observation that is naturally different from most of the data.
- Differences between groups of people in the dataset.

Therefore, an outlier should not automatically be removed before understanding why it exists.

#### 3. Which variables showed the clearest visual outliers?

BMI showed the clearest visual outliers in my analysis.

The box plot and scatter plot showed that most BMI values were concentrated in the normal range of the dataset, while some observations had much higher BMI values.

`PhysHlth` also contained high values such as 30 days, but this variable is limited to the number of poor physical health days in the last 30 days. Therefore, these values should not automatically be treated as outliers in the same way as BMI.

#### 4. How many observations were flagged by the Z-Score method?

The Z-score method identified **2,963 BMI observations** as potential outliers using the rule:

`|Z-score| > 3`

These observations were more than three standard deviations away from the average BMI.

#### 5. How many were flagged by the IQR method?

The IQR method identified **9,847 BMI observations** as potential outliers.

For BMI:

- Q1 = 24
- Q3 = 31
- IQR = 7
- Lower bound = 13.5
- Upper bound = 41.5

BMI values outside this range were identified as possible outliers.

#### 6. Why can Z-Score and IQR produce different results?

Z-score and IQR use different methods to identify unusual values.

Z-score uses the mean and standard deviation, while IQR uses the middle 50% of the dataset.

In my analysis, IQR identified more outliers because BMI values above 41.5 were flagged. The Z-score method was more conservative and mainly identified the more extreme BMI values.

All 2,963 Z-score outliers were also included among the IQR outliers.

#### 7. When should an outlier be retained?

An outlier should be retained when it represents a real and possible observation.

For example, a person may actually have a very high BMI. Even though the value is unusual, it does not necessarily mean that the data is incorrect.

Outliers can also contain useful information about rare or high-risk cases, so they should be investigated before being removed.

#### 8. When might removing an outlier be justified?

Removing an outlier may be justified when there is evidence that the value was caused by a data entry mistake, measurement error, or another data-quality problem.

Removal may also be considered when an extremely unusual value has a large effect on the analysis and there is a reasonable explanation for excluding it.

In my analysis, I used the more conservative Z-score method so that fewer potentially valid observations were removed.

#### 9. How did removal affect the descriptive statistics?

Before outlier treatment, the dataset contained **253,680 rows**. After removing the 2,963 extreme BMI observations, **250,717 rows** remained.

The average BMI changed only slightly:

- Before treatment: approximately **28.38**
- After treatment: approximately **28.02**

The standard deviation decreased:

- Before treatment: approximately **6.61**
- After treatment: approximately **5.61**

This shows that the extreme observations increased the spread of the BMI values. Removing them reduced the variability while keeping the average relatively similar.

#### 10. How could untreated outliers affect a downstream ML model?

Untreated outliers can affect some machine learning models because extremely large or small values may have too much influence on the model.

They can change the mean and standard deviation, affect data scaling, and make distance-based models treat extreme observations very differently from normal observations.

For some models, this can reduce prediction performance or cause the model to learn patterns that do not represent most of the dataset.

However, real outliers can also contain useful information. Therefore, they should be analyzed carefully instead of always being removed.

# **Final Validation and Submission Checklist**

Before submitting:

- [ ] All four required sections are in this single notebook.
- [ ] PEW uses `pd.melt()` or `DataFrame.melt()`.
- [ ] Billboard uses `melt()` and Pandas string manipulation.
- [ ] Cars demonstrates missing-value analysis, dropping, imputation, duplicates, and validation.
- [ ] Diabetes includes box plot, scatter plot, Z-Score, and IQR.
- [ ] Every required visualization has a meaningful title and axis labels.
- [ ] Reflection questions are answered in Markdown.
- [ ] Data is loaded using relative paths from a clearly named data folder.
- [ ] `README.md`, `requirements.txt`, and `.gitignore` are included.
- [ ] There are no unnecessary `!pip install` cells.
- [ ] You restarted the kernel and successfully ran **Run All** from top to bottom.

# **General Guidelines**

## Expected Observations

### PEW
- Income categories are encoded as column headings in the original wide dataset.
- `melt()` should preserve religion as an identifier while moving income categories into rows.
- The long form is easier to group, filter, and visualize.

### Billboard
- Weekly rankings are stored across repeated week columns.
- `melt()` converts weekly columns into observations.
- Week labels usually require string cleanup/extraction before numeric analysis.
- Missing rankings often indicate that a song was not ranked in that week and should not automatically be imputed.

### Cars
- Students should inspect before deleting the first row; removal should be evidence-based.
- Some numeric-looking columns may require conversion from `object`.
- Mean imputation is sensitive to skew/outliers; median can be more robust.
- Dropping every column containing a missing value is generally too destructive when missingness is small.

### Diabetes
- Box plots and IQR are closely related but students should understand that visualization and rule-based detection serve different purposes.
- Z-Score works best when a roughly normal-distribution assumption is reasonable.
- IQR is more robust for skewed distributions.
- Outliers should not be removed automatically without context.

## Common Mistakes

- Using absolute paths such as `C:\Users\...`.
- Treating `melt()` as equivalent to simply renaming columns.
- Cleaning Billboard week labels without converting the result to numeric.
- Filling every missing value with zero.
- Dropping data without first reporting the effect.
- Using `|z| > 2` when the assignment specifies `|z| > 3`.
- Computing IQR incorrectly or reversing the lower/upper bound formulas.
- Removing outliers from the original dataframe and losing the baseline for comparison.
- Producing plots without titles, labels, or interpretation.
- Adding `!pip install` cells even though the project includes `requirements.txt`.

## Suggested Instructor Solutions / Checks

- Accept both `pd.melt(df, ...)` and `df.melt(...)`.
- Accept logically equivalent Pandas string-cleaning approaches.
- For missing values, grade the quality of the justification, not only the specific method selected.
- Require evidence that the notebook runs sequentially.
- Confirm that each data file is loaded from the submitted project using relative paths.

## Stretch Challenges

1. Build reusable functions for tidying and cleaning.
2. Compare multiple imputation strategies quantitatively.
3. Use `ColumnTransformer` or a Scikit-Learn preprocessing pipeline.
4. Compare outlier treatment before/after scaling.
5. Investigate how removing outliers changes a simple regression model.
6. Export cleaned datasets to a `processed/` subfolder without overwriting raw data.